# 02. Exploratory Data Analysis

This notebook focuses on understanding the relationships in the dataset before modeling. We want to answer questions such as:

- Is the target variable balanced?
- Which features are most strongly associated with malignancy?
- Are there obvious outliers or redundant patterns?
- Which characteristics are worth investigating in the modeling stage?

We will inspect the data using descriptive statistics and visualizations, but we will avoid making medical claims beyond what the dataset can support.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
data_path = project_root / 'data' / 'data.csv'

df = pd.read_csv(data_path)
# Clean the known dataset artifacts before analysis.
df = df.drop(columns=[col for col in df.columns if 'Unnamed:' in str(col)], errors='ignore')
df = df.drop(columns=['id'], errors='ignore')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist()[:10], '...')
print('\nTarget distribution:')
print(df['diagnosis'].value_counts())

In [ ]:
# Encode the target for correlation analysis.
encoded = df.copy()
encoded['diagnosis_numeric'] = encoded['diagnosis'].map({'B': 0, 'M': 1})

# Correlation with the target.
correlations = encoded.drop(columns=['diagnosis']).corr()['diagnosis_numeric'].abs().sort_values(ascending=False)
print(correlations.head(10))

In [ ]:
# Distribution of the target variable.
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='diagnosis', palette=['#4C72B0', '#C44E52'])
plt.title('Diagnosis distribution')
plt.xlabel('Diagnosis')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot of feature distributions by diagnosis for a representative feature.
feature = 'radius_mean'
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='diagnosis', y=feature, palette=['#4C72B0', '#C44E52'])
plt.title(f'{feature} by diagnosis')
plt.xlabel('Diagnosis')
plt.ylabel(feature)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of the strongest correlations with the target.
selected = correlations.head(10).index.tolist()
subset = encoded[selected + ['diagnosis_numeric']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(subset, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation heatmap: strongest diagnostics-related features')
plt.tight_layout()
plt.show()

## Interpretation

From the dataset summary, we already know there are no missing values and no duplicate rows. The diagnosis target is imbalanced but not extremely skewed, which is manageable for modeling. Several geometric features such as radius, area, concavity, and perimeter are likely to be informative because they are known to vary noticeably between malignant and benign tumors in this dataset.

The key lesson from this notebook is that the dataset is strongly structured and suitable for supervised classification. We now move to preprocessing and train/test splitting so that we can model with a sound evaluation pipeline.